# Tutorial: Using MCP Tools with LangChain Agents

## Learning Objectives

By the end of this tutorial, you will be able to:
- Connect a LangGraph agent to an MCP server
- Use MCP tools within an agent workflow
- Understand the `MultiServerMCPClient` pattern

## Prerequisites

- Understanding of MCP concepts from the slides
- Familiarity with LangGraph agents and tool calling
- Basic async/await knowledge

## Setup

In [17]:
# !pip install langchain-mcp-adapters fastmcp langchain langchain-openai python-dotenv

In [18]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in your .env file"

print("Environment loaded!")

Environment loaded!


## The MCP Server

This directory contains `knowledge_base_server.py` - a FastMCP server with these tools:

| Tool | Description |
|------|-------------|
| `create_note` | Create a note with title, content, tags |
| `search_notes` | Search notes by keyword |
| `get_note` | Get a note by ID |
| `list_notes` | List all notes |
| `update_note` | Update a note |
| `delete_note` | Delete a note |
| `get_statistics` | Get KB statistics |

## Connecting to the MCP Server

Use `MultiServerMCPClient` to connect to MCP servers. For local servers, use `stdio` transport.

In [19]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# Path to our MCP server
SERVER_PATH = Path(".").resolve() / "knowledge_base_server.py"

# MCP client configuration
mcp_config = {
    "knowledge_base": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(SERVER_PATH)],
    }
}

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Running the Agent with MCP Tools

The pattern:
1. Create `MultiServerMCPClient` with server config
2. Get tools with `await client.get_tools()`
3. Pass tools to `create_agent()`

In [20]:
async def run_agent(user_message: str):
    """Run a LangChain agent with MCP tools."""
    client = MultiServerMCPClient(mcp_config)
    tools = await client.get_tools()
    
    print(f"Available tools: {[t.name for t in tools]}\n")
    
    agent = create_agent(model=llm, tools=tools)
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": user_message}]
    })
    
    return result["messages"][-1].content

## Test: Creating Notes

In [21]:
response = await run_agent(
    "Create a note titled 'Python Decorators' about how decorators work. "
    "Tag it with 'python' and 'tutorial'."
)
print(response)

Available tools: ['create_note', 'search_notes', 'get_note', 'list_notes', 'update_note', 'delete_note', 'get_statistics']

The note titled **"Python Decorators"** has been created successfully. Here are the details:

### Python Decorators
**Content:**
Decorators in Python are a powerful tool that allows you to modify the behavior of a function or class. They are often used to add functionality to existing code in a clean and readable way. A decorator is essentially a function that takes another function as an argument and extends its behavior without explicitly modifying it.

#### How Decorators Work:
1. **Function as First-Class Objects**: In Python, functions are first-class objects, meaning they can be passed around as arguments, returned from other functions, and assigned to variables.

2. **Defining a Decorator**: A decorator is defined as a function that returns another function. The inner function usually contains the code that extends or modifies the behavior of the original f

In [22]:
response = await run_agent(
    "Create a note about LangGraph basics covering nodes, edges, and state. "
    "Tag it with 'langgraph' and 'agents'."
)
print(response)

Available tools: ['create_note', 'search_notes', 'get_note', 'list_notes', 'update_note', 'delete_note', 'get_statistics']

The note titled **"LangGraph Basics"** has been successfully created. Here are the details:

### LangGraph Basics
LangGraph is a framework that allows for the representation and manipulation of knowledge in a graph structure. The fundamental components of LangGraph include:

1. **Nodes**: These are the individual entities or concepts within the graph. Each node can represent a specific piece of information, such as a word, phrase, or even a more complex idea. Nodes can have attributes that provide additional context or metadata.

2. **Edges**: Edges are the connections between nodes. They define the relationships between different entities in the graph. Edges can be directed or undirected, indicating the nature of the relationship (e.g., a parent-child relationship or a mutual connection).

3. **State**: The state in LangGraph refers to the current configuration o

## Test: Searching and Listing

In [23]:
response = await run_agent("Search for notes about Python.")
print(response)

Available tools: ['create_note', 'search_notes', 'get_note', 'list_notes', 'update_note', 'delete_note', 'get_statistics']

I found two notes related to Python:

1. **Title:** Python Decorators
   - **Content:** 
     Decorators in Python are a powerful tool that allows you to modify the behavior of a function or class. They are often used to add functionality to existing code in a clean and readable way. A decorator is essentially a function that takes another function as an argument and extends its behavior without explicitly modifying it.
     
     ### How Decorators Work:
     1. **Function as First-Class Objects**: In Python, functions are first-class objects, meaning they can be passed around as arguments, returned from other functions, and assigned to variables.
     
     2. **Defining a Decorator**: A decorator is defined as a function that returns another function. The inner function usually contains the code that extends the behavior of the original function.
     
     3. 

In [24]:
response = await run_agent("List all my notes and show me the knowledge base statistics.")
print(response)

Available tools: ['create_note', 'search_notes', 'get_note', 'list_notes', 'update_note', 'delete_note', 'get_statistics']

### Your Notes

1. **Title**: LangGraph Basics
   - **Content**: LangGraph is a framework that allows for the representation and manipulation of knowledge in a graph structure. The fundamental components of LangGraph include:
     - **Nodes**: Individual entities or concepts within the graph.
     - **Edges**: Connections between nodes that define relationships.
     - **State**: The current configuration of the graph.
   - **Tags**: langgraph, agents
   - **Created At**: December 10, 2025
   - **Updated At**: December 10, 2025

2. **Title**: Python Decorators
   - **Content**: Decorators in Python are a powerful tool that allows you to modify the behavior of a function or class. They are often used to add functionality to existing code in a clean and readable way. A decorator is essentially a function that takes another function as an argument and extends its beh

## Test: Multi-Tool Task

In [25]:
response = await run_agent(
    "Check what notes I have, then create a 'Learning Roadmap' note "
    "that references the topics from my existing notes. Tag it with 'meta'."
)
print(response)

Available tools: ['create_note', 'search_notes', 'get_note', 'list_notes', 'update_note', 'delete_note', 'get_statistics']

I have checked your existing notes and created a new note titled **"Learning Roadmap"**. This note references the topics from your existing notes and is tagged with **"meta"**. 

### Learning Roadmap
This roadmap outlines the key topics to explore based on existing notes:

1. **LangGraph Basics**
   - Understanding nodes, edges, and state in graph structures.
   - Applications in natural language processing and knowledge representation.

2. **Python Decorators**
   - How decorators work and their use cases.
   - Examples of defining and using decorators in Python.

This roadmap serves as a guide for further learning and exploration in these areas. 

If you need any further assistance or modifications, feel free to ask!


## Inspecting Tool Calls

Let's see the full trace of what the agent does.

In [26]:
async def run_agent_verbose(user_message: str):
    """Run agent and show detailed tool call trace."""
    client = MultiServerMCPClient(mcp_config)
    tools = await client.get_tools()
    agent = create_agent(model=llm, tools=tools)
    
    result = await agent.ainvoke({
        "messages": [{"role": "user", "content": user_message}]
    })
    
    print("TRACE:")
    print("=" * 50)
    for i, msg in enumerate(result["messages"]):
        msg_type = type(msg).__name__
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"[{i}] Tool: {tc['name']}({tc['args']})")
        elif msg_type == "ToolMessage":
            content = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
            print(f"[{i}] Result: {content}")
    print("=" * 50)
    
    return result["messages"][-1].content

In [27]:
response = await run_agent_verbose(
    "Find notes tagged 'python' and add 'programming' to their tags."
)
print(f"\nFinal response:\n{response}")

TRACE:
[1] Tool: list_notes({'tag_filter': 'python'})
[2] Result: [{'type': 'text', 'text': '{"status":"success","total_count":4,"returned_count":2,"filter_applied":"python","notes":[{"id":"309f851d","title":"Python Decorators","content":"Decorators in Python are a powerful tool that allows you to modify the behavior of a function or class. They are often used to add functionality to existing code in a clean and readable way. A decorator is essentially a function that takes another function as an argument and extends its behavior without explicitly modifying it.\\n\\n### How Decorators Work:\\n1. **Function as First-Class Objects**: In Python, functions are first-class objects, meaning they can be passed around as arguments, returned from other functions, and assigned to variables.\\n\\n2. **Defining a Decorator**: A decorator is defined as a function that returns another function. The inner function usually contains the code that extends or modifies the behavior of the original functi

## Cleanup

In [28]:
import json

storage_file = Path(".").resolve() / "data" / "notes.json"
if storage_file.exists():
    storage_file.write_text(json.dumps({"notes": {}}, indent=2))
    print("Knowledge base cleared.")

Knowledge base cleared.


## Summary

- Create `MultiServerMCPClient(config)` with `stdio` transport for local MCP servers
- Call `await client.get_tools()` to get LangChain-compatible tools
- Pass tools to `create_agent()` just like any other LangChain tools
- The same MCP server works with Claude Desktop, Cursor, and custom agents